In [1]:
import stim
import pymatching
import numpy as np
from itertools import combinations

QEC_CYCLE = 5
DISTANCE = 3
MAX_SYNDROME_WEIGHT = 3
BEFORE_MEAS_FLIP_P = 0.01

# Output files
LUT_SYND_FILE = "lut_synd.mem"   # hex lines, each is N_DET-bit syndrome packed into an integer
LUT_OUT_FILE  = "lut_out.mem"    # binary lines, each is 0 or 1 (predicted observable bit)

def main():
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        before_measure_flip_probability=BEFORE_MEAS_FLIP_P,
        distance=DISTANCE,
        rounds=QEC_CYCLE,
    )

    dem = circuit.detector_error_model(decompose_errors=True)
    matching = pymatching.Matching.from_detector_error_model(dem)
    num_det = dem.num_detectors

    if num_det != QEC_CYCLE * 8:
        print(f"[WARN] num_det={num_det} not equal to rounds*8={QEC_CYCLE*8}. "
              "Continuing anyway (your circuit settings might differ).")

    print(f"Generating LUT for d={DISTANCE}, rounds={QEC_CYCLE}")
    print(f"num_det={num_det}, max_weight={MAX_SYNDROME_WEIGHT}")

    patterns = []
    outs = []
    for w in range(MAX_SYNDROME_WEIGHT + 1):
        for locs in combinations(range(num_det), w):
            syndrome = np.zeros(num_det, dtype=np.uint8)
            for i in locs:
                syndrome[i] = 1

            pred = matching.decode(syndrome, algorithm="mwpm")
            outbit = int(pred[0] & 1)
            s_int = 0
            for i, b in enumerate(syndrome):
                if b:
                    s_int |= (1 << i)

            patterns.append(s_int)
            outs.append(outbit)

    print(f"LUT entries: {len(patterns)}")
    hex_digits = (num_det + 3) // 4

    with open(LUT_SYND_FILE, "w") as f:
        for p in patterns:
            f.write(f"{p:0{hex_digits}x}\n")

    # Write output bits (0/1), one per line
    with open(LUT_OUT_FILE, "w") as f:
        for b in outs:
            f.write(f"{b:d}\n")

    print(f"Wrote {LUT_SYND_FILE} and {LUT_OUT_FILE}")
    print("Done.")

if __name__ == "__main__":
    main()

Generating LUT for d=3, rounds=5
num_det=40, max_weight=3
LUT entries: 10701
Wrote lut_synd.mem and lut_out.mem
Done.


In [2]:
import stim
import numpy as np

QEC_CYCLE = 5
DISTANCE = 3
BEFORE_MEAS_FLIP_P = 0.01
SHOTS = 1000

VECTORS_FILE = "tb_vectors.mem"

def main():
    circuit = stim.Circuit.generated(
        "surface_code:rotated_memory_x",
        before_measure_flip_probability=BEFORE_MEAS_FLIP_P,
        distance=DISTANCE,
        rounds=QEC_CYCLE,
    )

    dem = circuit.detector_error_model(decompose_errors=True)
    num_det = dem.num_detectors
    hex_digits = (num_det + 3) // 4

    sampler = circuit.compile_detector_sampler()
    detectors, observables = sampler.sample(shots=SHOTS, separate_observables=True)

    # Pack each syndrome into hex, and write observable[0]
    with open(VECTORS_FILE, "w") as f:
        for d, o in zip(detectors, observables):
            s_int = 0
            # d is a numpy array of bits length num_det
            for i, b in enumerate(d):
                if int(b) & 1:
                    s_int |= (1 << i)

            obs0 = int(o[0] & 1)
            f.write(f"{s_int:0{hex_digits}x} {obs0:d}\n")

    print(f"Wrote {VECTORS_FILE} with {SHOTS} vectors")
    print(f"num_det={num_det} (hex digits {hex_digits})")

if __name__ == "__main__":
    main()

Wrote tb_vectors.mem with 1000 vectors
num_det=40 (hex digits 10)
